# Description

Analyzes drug-disease prediction differences between the gene-based and module-based approaches using the ARCHS4 CLAMP model.

For each drug-disease pair in the PharmacotherapyDB gold standard, it compares the standardized scores from both methods and identifies pairs where the methods disagree (different signs). It focuses on cardiovascular diseases and the Niacin case discussed in the PhenoPlier manuscript.

**Inputs** (from `021-prediction_performance.ipynb`):
- `predictions_results_aggregated.pkl`: averaged prediction scores per (trait, drug, method)

**Category note**: `indications-verbose.tsv` (PharmacotherapyDB) does not include the DM/SYM/NOT categorization from PhenoPlier's `pharmacotherapydb-v1.0/indications.tsv`. Category is approximated from `gold_standard.pkl`: `true_class=1 → DM`, `true_class=0 → NOT`.

# Module loading

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [10]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
assert DATA_DIR.exists()

PREDICTIONS_DIR = here('output/drug_disease_analyses') / 'predictions'
assert PREDICTIONS_DIR.exists()

# Data loading

## PharmacotherapyDB

### Gold standard set

In [11]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


### Drug and disease name lookup

Using `indications-verbose.tsv` from [PharmacotherapyDB](https://github.com/dhimmel/indications).
This file has one row per (doid_code, drugbank_id, resource) — we deduplicate to get unique drug/disease names.
Category (DM/SYM/NOT) is approximated from `true_class` in the gold standard.

In [12]:
indications = pd.read_csv(
    DATA_DIR / 'indications-verbose.tsv',
    sep='\t',
    usecols=['doid_code', 'drugbank_id', 'drugbank_name', 'doid_name'],
).drop_duplicates(subset=['doid_code', 'drugbank_id'])

print(f'Indications shape (after dedup): {indications.shape}')
display(indications.head())

Indications shape (after dedup): (13934, 4)


,doid_code,drugbank_id,drugbank_name,doid_name
0,DOID:0014667,DB00091,Cyclosporine,disease of metabolism
1,DOID:0014667,DB00117,L-Histidine,disease of metabolism
2,DOID:0014667,DB00121,Biotin,disease of metabolism
3,DOID:0014667,DB00151,L-Cysteine,disease of metabolism
4,DOID:0014667,DB00165,Pyridoxine,disease of metabolism


In [13]:
# Build gold_standard_info: gold_standard + disease/drug names + category
gold_standard_info = gold_standard.copy()
gold_standard_info['category'] = gold_standard_info['true_class'].map({1: 'DM', 0: 'NOT'})

# Add disease and drug names from indications-verbose
name_lookup = indications.set_index(['doid_code', 'drugbank_id'])

gold_standard_info = gold_standard_info.join(
    name_lookup.rename(columns={'doid_name': 'disease', 'drugbank_name': 'drug_name'}),
    on=['trait', 'drug'],
)

# Fill any missing names with the raw IDs
gold_standard_info['disease'] = gold_standard_info['disease'].fillna(gold_standard_info['trait'])
gold_standard_info['drug_name'] = gold_standard_info['drug_name'].fillna(gold_standard_info['drug'])

display(gold_standard_info.shape)
display(gold_standard_info.head())

(998, 6)

,trait,drug,true_class,category,drug_name,disease
0,DOID:10652,DB00843,1,DM,Donepezil,Alzheimer's disease
1,DOID:10652,DB00674,1,DM,Galantamine,Alzheimer's disease
2,DOID:10652,DB01043,1,DM,Memantine,Alzheimer's disease
3,DOID:10652,DB00989,1,DM,Rivastigmine,Alzheimer's disease
4,DOID:10652,DB00810,0,NOT,Biperiden,Alzheimer's disease


## Prediction results (aggregated)

In [14]:
predictions_avg = pd.read_pickle(PREDICTIONS_DIR / 'predictions_results_aggregated.pkl')
display(predictions_avg.shape)
display(predictions_avg.head())

(1370, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,Gene-based,294.2,1.0
1,DOID:0050741,DB00215,Module-based,313.8,1.0
2,DOID:0050741,DB00704,Gene-based,277.4,1.0
3,DOID:0050741,DB00704,Module-based,279.2,1.0
4,DOID:0050741,DB00822,Gene-based,536.6,1.0


### Merge with gold standard info

In [15]:
pharmadb_predictions = pd.merge(
    gold_standard_info,
    predictions_avg,
    on=['trait', 'drug'],
    how='inner',
)

In [16]:
pharmadb_predictions = pharmadb_predictions[
    ['trait', 'drug', 'disease', 'drug_name', 'method', 'score', 'true_class_x', 'category']
].rename(columns={'true_class_x': 'true_class'})

display(pharmadb_predictions.shape)
assert pharmadb_predictions.shape[0] == predictions_avg.shape[0]
display(pharmadb_predictions.head())

(1370, 8)

,trait,drug,disease,drug_name,method,score,true_class,category
0,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Gene-based,125.2,1,DM
1,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Module-based,441.6,1,DM
2,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Gene-based,298.6,1,DM
3,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Module-based,192.0,1,DM
4,DOID:10652,DB01043,Alzheimer's disease,Memantine,Gene-based,90.4,1,DM


In [17]:
print('Unique diseases:', pharmadb_predictions['disease'].nunique())
print('Unique drugs:', pharmadb_predictions['drug'].nunique())

Unique diseases: 71
Unique drugs: 337


In [18]:
data_stats = pharmadb_predictions.groupby('method')['score'].describe()
display(data_stats)

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
Gene-based,685.0,343.0,171.243071,8.8,200.2,343.6,479.6,670.9
Module-based,685.0,343.0,182.938962,1.2,188.8,334.4,484.8,681.2


# Standardize scores for each method

In [19]:
def _standardize(x):
    return (x['score'] - data_stats.loc[x['method'], 'mean']) / data_stats.loc[x['method'], 'std']


pharmadb_predictions = pharmadb_predictions.assign(
    score_std=pharmadb_predictions.apply(_standardize, axis=1)
)

display(pharmadb_predictions.head())

,trait,drug,disease,drug_name,method,score,true_class,category,score_std
0,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Gene-based,125.2,1,DM,-1.271876
1,DOID:10652,DB00843,Alzheimer's disease,Donepezil,Module-based,441.6,1,DM,0.538978
2,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Gene-based,298.6,1,DM,-0.259281
3,DOID:10652,DB00674,Alzheimer's disease,Galantamine,Module-based,192.0,1,DM,-0.825412
4,DOID:10652,DB01043,Alzheimer's disease,Memantine,Gene-based,90.4,1,DM,-1.475096


In [20]:
# Sanity check: standardized scores should be zero-mean, unit-variance per method
_tmp = pharmadb_predictions.groupby('method')[['score', 'score_std']].describe()
display(_tmp)

score                                                      \
              count   mean         std  min    25%    50%    75%    max   
method                                                                    
Gene-based    685.0  343.0  171.243071  8.8  200.2  343.6  479.6  670.9   
Module-based  685.0  343.0  182.938962  1.2  188.8  334.4  484.8  681.2   

             score_std                                                   \
                 count          mean  std       min       25%       50%   
method                                                                    
Gene-based       685.0  0.000000e+00  1.0 -1.951612 -0.833902  0.003504   
Module-based     685.0 -1.037289e-17  1.0 -1.868383 -0.842904 -0.047010   

                                  
                   75%       max  
method                            
Gene-based    0.797697  1.914822  
Module-based  0.775122  1.848704

# List diseases

In [21]:
pharmadb_predictions['disease'].unique()

array(["Alzheimer's disease", "Barrett's esophagus", "Crohn's disease",
       "Parkinson's disease", 'acquired immunodeficiency syndrome',
       'alcohol dependence', 'allergic rhinitis', 'anemia', 'DOID:2355',
       'ankylosing spondylitis', 'asthma', 'atherosclerosis', 'DOID:1936',
       'DOID:184', 'brain cancer', 'breast cancer', 'DOID:1612',
       'cervical cancer', 'DOID:784',
       'chronic obstructive pulmonary disease', 'coronary artery disease',
       'DOID:3393', 'epilepsy syndrome', 'DOID:1826', 'esophageal cancer',
       'gestational diabetes', 'DOID:1686', 'glaucoma', 'gout',
       'DOID:2531', 'hypertension', 'DOID:10763', 'hypothyroidism',
       'DOID:263', 'kidney cancer', 'liver cancer', 'DOID:1324',
       'lung cancer', 'malaria', 'DOID:12365', 'melanoma', 'migraine',
       'multiple sclerosis', 'nephrolithiasis', 'obesity',
       'osteoarthritis', 'osteoporosis', 'DOID:1793', 'pancreatic cancer',
       'pancreatitis', 'DOID:824', 'prostate cancer', 'DO

# Look for differences in scores between methods

For each drug-disease pair, identify where gene-based and module-based methods give opposite signs (one above mean, one below mean). These are the most interesting cases where the latent variable structure changes the prediction.

In [22]:
def _compare(x):
    """
    For a drug-disease pair with two rows (one per method), compute:
    - different_sign: whether the two standardized scores have opposite signs
    - score_difference: absolute difference between the two standardized scores
    """
    assert x.shape[0] == 2
    x_sign = np.sign(x['score_std'].values)
    x0 = x.iloc[0]['score_std']
    x1 = x.iloc[1]['score_std']
    return pd.Series(
        {'different_sign': x_sign[0] != x_sign[1], 'score_difference': np.abs(x0 - x1)}
    )

In [23]:
pharmadb_predictions = pharmadb_predictions.set_index(['trait', 'drug']).join(
    pharmadb_predictions.groupby(['trait', 'drug']).apply(_compare)
)

display(pharmadb_predictions.head())

/tmp/ipykernel_511633/932704532.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pharmadb_predictions.groupby(['trait', 'drug']).apply(_compare)


disease    drug_name        method  score  \
trait      drug                                                             
DOID:10652 DB00843  Alzheimer's disease    Donepezil    Gene-based  125.2   
           DB00843  Alzheimer's disease    Donepezil  Module-based  441.6   
           DB00674  Alzheimer's disease  Galantamine    Gene-based  298.6   
           DB00674  Alzheimer's disease  Galantamine  Module-based  192.0   
           DB01043  Alzheimer's disease    Memantine    Gene-based   90.4   

                    true_class category  score_std  different_sign  \
trait      drug                                                      
DOID:10652 DB00843           1       DM  -1.271876            True   
           DB00843           1       DM   0.538978            True   
           DB00674           1       DM  -0.259281           False   
           DB00674           1       DM  -0.825412           False   
           DB01043           1       DM  -1.475096           False   

                    score_difference  
trait      drug                       
DOID:10652 DB00843          1.810854  
           DB00843          1.810854  
           DB00674          0.566131  
           DB00674          0.566131  
           DB01043          0.473669

## Across all diseases

In [24]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[pharmadb_predictions['different_sign']].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.shape)
    display(_tmp)

(416, 9)

disease  \
trait      drug                                             
DOID:2377  DB01204                     multiple sclerosis   
           DB01204                     multiple sclerosis   
DOID:2841  DB01234                                 asthma   
           DB01234                                 asthma   
           DB00443                                 asthma   
           DB00443                                 asthma   
DOID:7148  DB00374                   rheumatoid arthritis   
           DB00374                   rheumatoid arthritis   
DOID:9008  DB01234                    psoriatic arthritis   
           DB01234                    psoriatic arthritis   
           DB00443                    psoriatic arthritis   
           DB00443                    psoriatic arthritis   
DOID:7147  DB00563                 ankylosing spondylitis   
           DB00563                 ankylosing spondylitis   
DOID:9074  DB00741           systemic lupus erythematosus   
           DB00741           systemic lupus erythematosus   
DOID:8893  DB00563                              psoriasis   
           DB00563                              psoriasis   
           DB00620                              psoriasis   
           DB00620                              psoriasis   
DOID:7148  DB00563                   rheumatoid arthritis   
           DB00563                   rheumatoid arthritis   
DOID:8893  DB01234                              psoriasis   
           DB01234                              psoriasis   
           DB00443                              psoriasis   
           DB00443                              psoriasis   
DOID:2377  DB00563                     multiple sclerosis   
           DB00563                     multiple sclerosis   
DOID:7147  DB01234                 ankylosing spondylitis   
           DB01234                 ankylosing spondylitis   
           DB00443                 ankylosing spondylitis   
           DB00443                 ankylosing spondylitis   
DOID:2377  DB01234                     multiple sclerosis   
           DB01234                     multiple sclerosis   
           DB00443                     multiple sclerosis   
           DB00443                     multiple sclerosis   
DOID:9074  DB01234           systemic lupus erythematosus   
           DB01234           systemic lupus erythematosus   
           DB00443           systemic lupus erythematosus   
           DB00443           systemic lupus erythematosus   
DOID:2377  DB00993                     multiple sclerosis   
           DB00993                     multiple sclerosis   
DOID:1793  DB00445                      pancreatic cancer   
           DB00445                      pancreatic cancer   
           DB00997                      pancreatic cancer   
           DB00997                      pancreatic cancer   
DOID:2841  DB00620                                 asthma   
           DB00620                                 asthma   
DOID:3571  DB00445                           liver cancer   
           DB00445                           liver cancer   
           DB00997                           liver cancer   
           DB00997                           liver cancer   
DOID:9008  DB00620                    psoriatic arthritis   
           DB00620                    psoriatic arthritis   
DOID:2377  DB00242                     multiple sclerosis   
           DB00242                     multiple sclerosis   
DOID:3393  DB00627                coronary artery disease   
           DB00627                coronary artery disease   
DOID:2377  DB00953                     multiple sclerosis   
           DB00953                     multiple sclerosis   
DOID:2841  DB00537                                 asthma   
           DB00537                                 asthma   
           DB01222                                 asthma   
           DB01222                                 asthma   
DOID:2355  DB01234      

In [25]:
def find_differences(trait_name):
    """Show drug-disease pairs for a given disease where methods disagree."""
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
        _tmp = pharmadb_predictions[
            (pharmadb_predictions['disease'] == trait_name)
            & (pharmadb_predictions['different_sign'])
        ].sort_values(
            ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
        )
        display(_tmp)

# Cardiovascular diseases

Niacin and cardiovascular traits — a key case study in the PhenoPlier manuscript.

## Coronary artery disease

In [26]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[
        (pharmadb_predictions['drug_name'] == 'Niacin')
        & (pharmadb_predictions['disease'] == 'coronary artery disease')
    ].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.head(50))

disease drug_name        method  score  \
trait     drug                                                              
DOID:3393 DB00627  coronary artery disease    Niacin  Module-based  161.8   
          DB00627  coronary artery disease    Niacin    Gene-based  546.4   

                   true_class category  score_std  different_sign  \
trait     drug                                                      
DOID:3393 DB00627           1       DM  -0.990494            True   
          DB00627           1       DM   1.187785            True   

                   score_difference  
trait     drug                       
DOID:3393 DB00627           2.17828  
          DB00627           2.17828

In [27]:
find_differences('coronary artery disease')

disease     drug_name        method  score  \
trait     drug                                                                  
DOID:3393 DB00627  coronary artery disease        Niacin  Module-based  161.8   
          DB00627  coronary artery disease        Niacin    Gene-based  546.4   
          DB01241  coronary artery disease   Gemfibrozil  Module-based  188.8   
          DB01241  coronary artery disease   Gemfibrozil    Gene-based  500.0   
          DB01098  coronary artery disease  Rosuvastatin  Module-based  509.6   
          DB01098  coronary artery disease  Rosuvastatin    Gene-based  225.4   
          DB00178  coronary artery disease      Ramipril  Module-based  419.2   
          DB00178  coronary artery disease      Ramipril    Gene-based  331.6   

                   true_class category  score_std  different_sign  \
trait     drug                                                      
DOID:3393 DB00627           1       DM  -0.990494            True   
          DB00627           1       DM   1.187785            True   
          DB01241           1       DM  -0.842904            True   
          DB01241           1       DM   0.916825            True   
          DB01098           1       DM   0.910686            True   
          DB01098           1       DM  -0.686743            True   
          DB00178           1       DM   0.416532            True   
          DB00178           1       DM  -0.066572            True   

                   score_difference  
trait     drug                       
DOID:3393 DB00627          2.178280  
          DB00627          2.178280  
          DB01241          1.759730  
          DB01241          1.759730  
          DB01098          1.597429  
          DB01098          1.597429  
          DB00178          0.483104  
          DB00178          0.483104

## Atherosclerosis

In [28]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'max_colwidth', None):
    _tmp = pharmadb_predictions[
        (pharmadb_predictions['drug_name'] == 'Niacin')
        & (pharmadb_predictions['disease'] == 'atherosclerosis')
    ].sort_values(
        ['score_difference', 'drug_name', 'method'], ascending=[False, False, False]
    )
    display(_tmp.head(50))

disease drug_name        method  score  true_class  \
trait     drug                                                                  
DOID:1936 DB00627  atherosclerosis    Niacin  Module-based  116.0           1   
          DB00627  atherosclerosis    Niacin    Gene-based  192.2           1   

                  category  score_std  different_sign  score_difference  
trait     drug                                                           
DOID:1936 DB00627       DM  -1.240851           False          0.360231  
          DB00627       DM  -0.880620           False          0.360231

In [29]:
find_differences('atherosclerosis')

,,disease,drug_name,method,score,true_class,category,score_std,different_sign,score_difference
trait,drug,,,,,,,,,
